# 🤖 Asistente Automatizado de Mensajes con IA

> **Autora:** Yasmin Beltre | Customer Success & Operations Specialist  
> **Curso:** Inteligencia Artificial — INDOTEL/BID/CYMETRIA 2026 — Módulo 8  
> **Técnicas:** NLP · TF-IDF · Naive Bayes · Gradio

---

Este proyecto implementa un asistente inteligente que **clasifica mensajes entrantes** y genera **respuestas automáticas** según la intención detectada. El flujo completo es:

1. Dataset de mensajes etiquetados por categoría
2. Limpieza de texto con NLP (NLTK)
3. Vectorización con TF-IDF
4. Clasificación con Naive Bayes
5. Generación de respuesta automática
6. Interfaz de chat interactiva con Gradio

## ⚙️ Paso 1 — Instalación de librerías

In [ ]:
!pip install nltk scikit-learn gradio pandas

## 📦 Paso 2 — Importaciones y recursos NLP

In [ ]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import gradio as gr

# Descargar recursos de NLTK
nltk.download('stopwords')

print("Librerías cargadas correctamente")

## 📋 Paso 3 — Dataset de mensajes

Se definen 29 mensajes de ejemplo en español, clasificados en 8 categorías: saludo, información, horario, precios, soporte, agradecimiento, despedida y consulta.

In [ ]:
datos = {
    "mensaje": [
        "Hola", "Buenos días", "Buenas tardes", "Hola cómo estás",
        "Necesito información", "Quiero conocer información", "Deseo más detalles", "Necesito detalles del servicio",
        "Quiero conocer los horarios", "Cuál es el horario", "A qué hora abren", "Cuándo trabajan",
        "Cuáles son los precios", "Necesito saber precios", "Quiero ver costos", "Cuánto cuesta",
        "Necesito ayuda", "Tengo un problema", "Requiero soporte", "Necesito soporte técnico",
        "Gracias", "Muchas gracias", "Gracias por ayudarme",
        "Hasta luego", "Adiós", "Nos vemos",
        "Quiero hacer una consulta", "Tengo una pregunta", "Necesito hacer una consulta"
    ],
    "categoria": [
        "saludo", "saludo", "saludo", "saludo",
        "informacion", "informacion", "informacion", "informacion",
        "horario", "horario", "horario", "horario",
        "precios", "precios", "precios", "precios",
        "soporte", "soporte", "soporte", "soporte",
        "agradecimiento", "agradecimiento", "agradecimiento",
        "despedida", "despedida", "despedida",
        "consulta", "consulta", "consulta"
    ]
}

df = pd.DataFrame(datos)
print(f"Dataset cargado: {len(df)} mensajes en {df['categoria'].nunique()} categorías")
print(df['categoria'].value_counts())

## 🧹 Paso 4 — Limpieza de texto con NLP

Se aplica preprocesamiento estándar: conversión a minúsculas, eliminación de caracteres especiales y remoción de palabras vacías (*stopwords*) en español.

In [ ]:
stop_words = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    return " ".join(palabras)

df["mensaje_limpio"] = df["mensaje"].apply(limpiar_texto)

print("Ejemplo de limpieza de texto:")
print(df[["mensaje", "mensaje_limpio"]].head(8).to_string(index=False))

## 🤖 Paso 5 — Vectorización TF-IDF y entrenamiento del modelo

**TF-IDF** (Term Frequency-Inverse Document Frequency) convierte el texto en vectores numéricos ponderando la importancia de cada palabra. **Naive Bayes Multinomial** es el clasificador — ideal para texto corto por su eficiencia y buen desempeño con vocabularios pequeños.

In [ ]:
vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(df["mensaje_limpio"])
y = df["categoria"]

modelo = MultinomialNB()
modelo.fit(X, y)

print("Modelo entrenado correctamente")
print(f"Vocabulario: {len(vectorizador.vocabulary_)} términos únicos")

## 💬 Paso 6 — Respuestas automáticas y función del asistente

Se define un diccionario de respuestas por categoría y la función `asistente()` que integra todo el pipeline: limpieza → vectorización → predicción → respuesta.

In [ ]:
respuestas = {
    "saludo":         "Hola, ¿cómo puedo ayudarte?",
    "informacion":    "Gracias por comunicarte. ¿Qué información necesitas?",
    "horario":        "Nuestro horario es de lunes a viernes de 8:00 a.m. a 5:00 p.m.",
    "precios":        "Puedes consultar nuestros precios en nuestro catálogo.",
    "soporte":        "Nuestro equipo de soporte revisará tu solicitud a la brevedad.",
    "agradecimiento": "Con gusto, estamos para ayudarte.",
    "despedida":      "Gracias por contactarnos. Hasta luego.",
    "consulta":       "Por favor indícanos tu consulta y te responderemos."
}

def asistente(mensaje):
    mensaje_limpio = limpiar_texto(mensaje)
    mensaje_vector = vectorizador.transform([mensaje_limpio])
    categoria = modelo.predict(mensaje_vector)[0]
    respuesta = respuestas.get(categoria, "No pude comprender tu solicitud. ¿Puedes reformularla?")
    return categoria, respuesta

# Prueba rápida
for prueba in ["Necesito soporte técnico", "Cuál es el horario", "Muchas gracias"]:
    cat, resp = asistente(prueba)
    print(f"Mensaje: '{prueba}'")
    print(f"  → Categoría: {cat} | Respuesta: {resp}\n")

## 🖥️ Paso 7 — Interfaz de chat interactiva con Gradio

Se construye una interfaz tipo chat usando `gr.Blocks` que muestra el historial de conversación, la categoría detectada y la respuesta generada en tiempo real.

In [ ]:
def responder_chat(mensaje, historial):
    categoria, respuesta = asistente(mensaje)
    respuesta_final = f"📌 Categoría: {categoria}\n🤖 {respuesta}"
    historial = historial + [
        {"role": "user", "content": mensaje},
        {"role": "assistant", "content": respuesta_final}
    ]
    return "", historial

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Asistente Automatizado de Mensajes")
    gr.Markdown("Asistente inteligente para clasificación y respuesta automática de mensajes en español.")

    chatbot = gr.Chatbot(type="messages", height=400)
    mensaje = gr.Textbox(placeholder="Escribe un mensaje aquí...", label="Tu mensaje")

    mensaje.submit(
        responder_chat,
        inputs=[mensaje, chatbot],
        outputs=[mensaje, chatbot]
    )

demo.launch(share=True)

## 💼 Aplicaciones prácticas en Customer Success

| Caso de uso | Beneficio |
|---|---|
| Respuestas a preguntas frecuentes | Reducción del tiempo de atención |
| Atención fuera de horario | Cobertura 24/7 automatizada |
| Escalación inteligente | Detección de casos que requieren atención humana |
| Consistencia en la comunicación | Respuestas estandarizadas y de calidad |

---
*Proyecto desarrollado como parte del curso de Inteligencia Artificial — INDOTEL/BID/CYMETRIA 2026*